In [ ]:
!pip install -q diffusers accelerate h5py tqdm

In [ ]:
DATASET = "cifar10"  # "cifar10" or "celeba"
NUM_SAMPLES = 20000
BATCH_SIZE = 500
OUTPUT_DIR = "outputs"

In [ ]:
import os
import h5py
import numpy as np
import torch
from diffusers import DDPMPipeline
from PIL import Image
from tqdm.auto import tqdm

MODEL_CONFIGS = {
    "cifar10": ("google/ddpm-cifar10-32", 32, None),
    "celeba": ("google/ddpm-celebahq-256", 256, 64),
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def generate_samples(pipeline, num_samples, batch_size, target_size=None):
    samples = []
    num_batches = (num_samples + batch_size - 1) // batch_size
    generated = 0

    for _ in tqdm(range(num_batches), desc="Generating"):
        current_batch = min(batch_size, num_samples - generated)
        if current_batch <= 0:
            break

        with torch.no_grad():
            if target_size:
                images = pipeline(batch_size=current_batch, output_type="pil").images
                batch_np = np.stack([
                    np.array(img.resize((target_size, target_size), Image.LANCZOS)).astype(np.float32) / 255.0
                    for img in images
                ], axis=-1)
            else:
                images = pipeline(batch_size=current_batch, output_type="np").images
                batch_np = np.transpose(images, (1, 2, 3, 0))

        samples.append(batch_np)
        generated += current_batch

    return np.concatenate(samples, axis=-1)[:, :, :, :num_samples].astype(np.float32)

In [ ]:
# Load and generate
model_id, native_res, target_size = MODEL_CONFIGS[DATASET]
batch_size = min(BATCH_SIZE, 16) if native_res >= 256 else BATCH_SIZE

print(f"Loading {model_id}...")
pipeline = DDPMPipeline.from_pretrained(model_id).to(device)

print(f"Generating {NUM_SAMPLES} samples...")
samples = generate_samples(pipeline, NUM_SAMPLES, batch_size, target_size)

In [ ]:
# Save to H5
output_path = f"{OUTPUT_DIR}/{DATASET.upper()}/DDPM/generated_images.h5"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with h5py.File(output_path, "w") as f:
    f.create_dataset("samples", data=samples, compression="gzip")

print(f"Saved to {output_path}")
print(f"Shape: {samples.shape}, Range: [{samples.min():.3f}, {samples.max():.3f}]")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = np.transpose(samples[:, :, :, i], (0, 1, 2))  # H, W, C
    ax.imshow(img)
    ax.axis("off")
plt.suptitle(f"DDPM {DATASET.upper()} Samples")
plt.tight_layout()
plt.show()

In [ ]:
try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print(f"Not in Colab. File saved at: {output_path}")